# Autovalores, autovetores e SVD

[Abrir este capítulo no Google Colab](https://colab.research.google.com/github/vnikoofard/calculo-numerico-livro/blob/main/livro/08-autovalores-svd.ipynb)

> **Objetivos.** Ao final deste capítulo você será capaz de:
> 1. interpretar autovalores e autovetores como as direções que uma matriz só estica, e calcular modos de vibração de uma estrutura;
> 2. usar o método da potência e o da potência inversa, e prever a sua velocidade;
> 3. saber o que as bibliotecas fazem (algoritmo QR) e por que não se usa o polinômio característico;
> 4. usar a decomposição em valores singulares (SVD) para aproximar uma matriz por outra de posto baixo;
> 5. verificar autovalores com o quociente de Rayleigh e com a forma dos modos.

In [ ]:
# Preparação do ambiente: funciona no seu computador, no Colab e no navegador (JupyterLite).
import sys
try:
    import cnlivro
except ImportError:
    if sys.platform == "emscripten":  # JupyterLite: Python rodando no navegador
        import micropip
        await micropip.install(["plotly", "scipy",
            "https://vnikoofard.github.io/calculo-numerico-livro/pacote/cnlivro-0.1.0-py3-none-any.whl"])
    else:  # Colab ou instalação local sem o pacote
        %pip install -q git+https://github.com/vnikoofard/calculo-numerico-livro

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from cnlivro.estilo import aplicar_estilo, mostrar, grafico_convergencia, grafico_iteracoes, com_controle, numero, CORES
from cnlivro import raizes
from cnlivro.verificacao import ordem_observada, ordem_iterativa, compara

aplicar_estilo()

In [ ]:
import scipy.linalg as sl
from cnlivro.autovalores import metodo_potencia, predio_cisalhamento

## 1. Problema de engenharia: um prédio e uma máquina no telhado

Um prédio de quatro andares vai receber, no telhado, um conjunto de ventiladores industriais que gira a **150 rpm**, ou seja, $2{,}5$ Hz. Se uma das frequências naturais do prédio estiver perto de $2{,}5$ Hz, a vibração da máquina entra em **ressonância** com a estrutura, e as amplitudes podem ficar perigosas. As normas pedem uma distância de pelo menos 20 % entre a frequência de operação e a frequência natural mais baixa, a **fundamental**.

O modelo mais simples é o "prédio de cisalhamento": cada laje é uma massa $m_i$, e os pilares entre duas lajes funcionam como uma mola horizontal de rigidez $k_i$. A segunda lei de Newton para os deslocamentos horizontais $\mathbf{u}(t)$ dos andares dá

$$
M\ddot{\mathbf{u}} + K\mathbf{u} = \mathbf{0},
$$

com $M$ diagonal (as massas) e $K$ tridiagonal (as rigidezes). Procurando soluções em que todos os andares oscilam juntos, $\mathbf{u}(t) = \mathbf{v}\cos(\omega t)$, chegamos a

$$
K\mathbf{v} = \omega^2 M\mathbf{v}.
$$ (eq-modos)

É um **problema de autovalores**: os autovalores $\lambda = \omega^2$ dão as frequências naturais $f = \omega/2\pi$, e os autovetores $\mathbf{v}$ dão os **modos de vibração**, a forma com que o prédio balança em cada frequência.

In [ ]:
massas = [30e3, 30e3, 30e3, 20e3]          # kg, do 1º andar ao telhado
rigidezes = [60e6, 50e6, 40e6, 30e6]       # N/m, do térreo ao último andar
M, K = predio_cisalhamento(massas, rigidezes)
print("M (t) =\n", M / 1e3)
print("K (MN/m) =\n", K / 1e6)

## 2. A ideia em uma imagem

Uma matriz $2 \times 2$ transforma o círculo unitário numa elipse. Quase todo vetor muda de **direção**. Os **autovetores** são as direções especiais que a matriz só estica (ou encolhe, ou inverte), pelo fator $\lambda$. Os **vetores singulares** são outra coisa: são os eixos da elipse, as direções de maior e menor esticamento.

In [ ]:
th = np.linspace(0, 2 * np.pi, 300)
circulo = np.vstack([np.cos(th), np.sin(th)])
matrizes = [("simétrica: autovetores = eixos da elipse", np.array([[2.0, 0.8], [0.8, 1.0]])),
            ("não simétrica: autovetores ≠ eixos", np.array([[1.5, 1.2], [0.1, 0.7]]))]
fig = make_subplots(rows=1, cols=2, subplot_titles=[m[0] for m in matrizes], horizontal_spacing=0.08)
for j, (nome, A) in enumerate(matrizes, start=1):
    elipse = A @ circulo
    fig.add_trace(go.Scatter(x=circulo[0], y=circulo[1], mode="lines", name="círculo unitário",
                             line=dict(color=CORES["referencia"], dash="dot"), showlegend=(j == 1)), 1, j)
    fig.add_trace(go.Scatter(x=elipse[0], y=elipse[1], mode="lines", name="imagem A·(círculo)",
                             line=dict(color=CORES["exata"]), showlegend=(j == 1)), 1, j)
    lam, V = np.linalg.eig(A)
    for k in range(2):
        v = V[:, k].real
        Av = A @ v
        fig.add_trace(go.Scatter(x=[0, v[0]], y=[0, v[1]], mode="lines", line=dict(color=CORES["aproximacao"], width=2),
                                 name="autovetor v", showlegend=(j == 1 and k == 0)), 1, j)
        fig.add_trace(go.Scatter(x=[0, Av[0]], y=[0, Av[1]], mode="lines", line=dict(color=CORES["aproximacao"], width=5),
                                 opacity=0.45, name="A v = λ v", showlegend=(j == 1 and k == 0)), 1, j)
        fig.add_annotation(x=Av[0], y=Av[1], text=f"λ = {numero(lam[k].real, 2)}", showarrow=False, yshift=12,
                           font=dict(color=CORES["aproximacao"]), row=1, col=j)
    U, s, Vt = np.linalg.svd(A)
    for k in range(2):
        eixo = s[k] * U[:, k]
        fig.add_trace(go.Scatter(x=[-eixo[0], eixo[0]], y=[-eixo[1], eixo[1]], mode="lines",
                                 line=dict(color=CORES["erro"], dash="dash"), name="eixos (valores singulares σ)",
                                 showlegend=(j == 1 and k == 0)), 1, j)
fig.update_xaxes(range=[-2.6, 2.6], zeroline=False)
fig.update_yaxes(range=[-2.0, 2.0], zeroline=False)
fig.update_yaxes(scaleanchor="x", row=1, col=1)
fig.update_yaxes(scaleanchor="x2", row=1, col=2)
fig.update_layout(width=980, height=440, legend=dict(orientation="h", y=-0.12), margin=dict(t=60))
mostrar(fig)

**O que notar:** à esquerda, a matriz é simétrica, e os autovetores (azul) coincidem com os eixos da elipse (vermelho tracejado). As matrizes de rigidez e de massa das estruturas são simétricas, e por isso os modos de vibração são ortogonais e bem comportados. À direita, a matriz não é simétrica: os autovetores nem são perpendiculares, e os eixos da elipse, dados pela SVD, são outra coisa.

## 3. Formulação matemática

### 3.1 Autovalores e autovetores

Um número $\lambda$ e um vetor $\mathbf{v} \ne \mathbf{0}$ com $A\mathbf{v} = \lambda\mathbf{v}$ são um **autovalor** e um **autovetor** de $A$. Como $(A - \lambda I)\mathbf{v} = \mathbf{0}$ tem solução não nula, $\det(A - \lambda I) = 0$: os autovalores são as raízes do **polinômio característico**, de grau $n$. Essa caracterização é ótima para matrizes $2 \times 2$ e para a teoria, e péssima para calcular (seção 7.2).

Se $A$ é **simétrica**, tudo fica melhor: os autovalores são reais, e há uma base de autovetores **ortonormais**, $A = Q\Lambda Q^{\mathsf T}$. No problema generalizado {eq}`eq-modos`, com $K$ simétrica e $M$ simétrica positiva definida, os autovalores são reais e positivos, e os modos são ortogonais "com peso $M$": $\mathbf{v}_i^{\mathsf T} M \mathbf{v}_j = 0$ para $i \ne j$.

### 3.2 O método da potência

Escreva um vetor qualquer na base de autovetores, $\mathbf{x}_0 = c_1\mathbf{v}_1 + \cdots + c_n\mathbf{v}_n$, com $|\lambda_1| > |\lambda_2| \ge \cdots$. Multiplicando por $A$ repetidas vezes,

$$
A^k\mathbf{x}_0 = \lambda_1^k\left(c_1\mathbf{v}_1 + c_2\Big(\frac{\lambda_2}{\lambda_1}\Big)^k\mathbf{v}_2 + \cdots\right).
$$

Todos os termos, menos o primeiro, encolhem. Normalizando a cada passo para evitar overflow, $\mathbf{x}_k$ aponta cada vez mais na direção de $\mathbf{v}_1$, o autovetor do autovalor de **maior módulo**. O erro cai pelo fator $|\lambda_2/\lambda_1|$ a cada passo: a convergência é **linear**, e lenta quando os dois maiores autovalores são próximos.

O autovalor é estimado pelo **quociente de Rayleigh**,

$$
\rho(\mathbf{x}) = \frac{\mathbf{x}^{\mathsf T}A\mathbf{x}}{\mathbf{x}^{\mathsf T}\mathbf{x}}
\qquad\text{ou, no problema generalizado,}\qquad
\rho(\mathbf{x}) = \frac{\mathbf{x}^{\mathsf T}K\mathbf{x}}{\mathbf{x}^{\mathsf T}M\mathbf{x}} .
$$ (eq-rayleigh)

Para matrizes simétricas, um erro $\epsilon$ no autovetor dá um erro de apenas $\epsilon^2$ no autovalor. Além disso, vale o **princípio de Rayleigh**: para **qualquer** vetor $\mathbf{x}$,

$$
\lambda_{\min} \le \rho(\mathbf{x}) \le \lambda_{\max}.
$$

No prédio, qualquer forma de deslocamento que você imagine dá uma estimativa **por cima** de $\omega_1^2$. É uma verificação poderosa (seção 6).

### 3.3 Potência inversa e deslocamentos

Os autovalores de $A^{-1}$ são $1/\lambda_i$. Aplicar o método da potência a $A^{-1}$, resolvendo $A\mathbf{y} = \mathbf{x}_k$ a cada passo com uma fatoração LU feita uma vez só (capítulo 4), converge para o autovalor de **menor** módulo. No prédio, é a frequência **fundamental**, a que interessa.

Mais geralmente, a potência inversa com **deslocamento** $\sigma$ usa $(A - \sigma I)^{-1}$ e converge para o autovalor mais próximo de $\sigma$, com fator $|\lambda_j - \sigma|/|\lambda_k - \sigma|$. Com $\sigma$ bem escolhido, a convergência é muito rápida.

### 3.4 O que as bibliotecas fazem: o algoritmo QR

Para calcular **todos** os autovalores, as bibliotecas usam o **algoritmo QR**. A ideia básica é repetir

$$
A_k = Q_k R_k, \qquad A_{k+1} = R_k Q_k = Q_k^{\mathsf T} A_k Q_k .
$$

Cada $A_{k+1}$ é semelhante a $A_k$ e tem os mesmos autovalores. A sequência converge para uma matriz triangular, com os autovalores na diagonal. As versões práticas primeiro reduzem $A$ à forma tridiagonal (se simétrica) ou de Hessenberg, e usam deslocamentos, com custo total de $O(n^3)$. É o que fazem `np.linalg.eigh` e `scipy.linalg.eigh`. Para matrizes grandes e esparsas, das quais só se querem alguns autovalores, há métodos de Lanczos e Arnoldi (`scipy.sparse.linalg.eigsh`).

### 3.5 A decomposição em valores singulares

Toda matriz $A$, $m \times n$, quadrada ou não, pode ser escrita como

$$
A = U\Sigma V^{\mathsf T} = \sum_{i=1}^{r} \sigma_i\, \mathbf{u}_i \mathbf{v}_i^{\mathsf T},
$$ (eq-svd)

com $U$ e $V$ ortogonais e $\sigma_1 \ge \sigma_2 \ge \cdots \ge \sigma_r > 0$ os **valores singulares**. Geometricamente, $V^{\mathsf T}$ gira, $\Sigma$ estica ao longo dos eixos e $U$ gira de novo. Os $\sigma_i$ são os semieixos da elipse da figura, e $\kappa_2(A) = \sigma_1/\sigma_r$. Os $\sigma_i^2$ são os autovalores de $A^{\mathsf T}A$.

O resultado mais útil é o **teorema de Eckart-Young**: a melhor aproximação de $A$ por uma matriz de posto $k$ é obtida truncando a soma {eq}`eq-svd` nos $k$ primeiros termos, e o erro é

$$
\|A - A_k\|_2 = \sigma_{k+1}.
$$

Se os valores singulares caem depressa, poucos termos guardam quase toda a informação. Isso está por trás da compressão de dados (seção 5.2), da **análise de componentes principais** (PCA), que é a SVD de uma tabela de dados centrada, e da regularização de mínimos quadrados (capítulo 7).

## 4. À mão

Um prédio de dois andares com massas e rigidezes iguais, $m = 1$ e $k = 1$ (em unidades quaisquer), tem

$$
K = \begin{bmatrix} 2 & -1 \\ -1 & 1 \end{bmatrix}, \qquad M = I .
$$

**Polinômio característico.** $\det(K - \lambda I) = (2 - \lambda)(1 - \lambda) - 1 = \lambda^2 - 3\lambda + 1 = 0$, então $\lambda = (3 \pm \sqrt5)/2$: $\lambda_1 = 0{,}382$ e $\lambda_2 = 2{,}618$. Os modos são $\mathbf{v}_1 = (0{,}618;\ 1)$, com os dois andares para o mesmo lado, e $\mathbf{v}_2 = (-1{,}618;\ 1)$, com os andares em oposição.

**Potência inversa.** Para achar o modo fundamental sem o polinômio, iteramos com $K^{-1} = \begin{bmatrix} 1 & 1 \\ 1 & 2 \end{bmatrix}$, normalizando pela última componente:

| $k$ | $\mathbf{x}_k$ | $K^{-1}\mathbf{x}_k$ | normalizado |
| --- | --- | --- | --- |
| 0 | $(1;\ 1)$ | $(2;\ 3)$ | $(0{,}667;\ 1)$ |
| 1 | $(0{,}667;\ 1)$ | $(1{,}667;\ 2{,}667)$ | $(0{,}625;\ 1)$ |
| 2 | $(0{,}625;\ 1)$ | $(1{,}625;\ 2{,}625)$ | $(0{,}619;\ 1)$ |

O vetor converge para $(0{,}618;\ 1)$. O erro na primeira componente cai de $0{,}049$ para $0{,}007$ e depois para $0{,}001$, por um fator de cerca de 7 por passo, que é $\lambda_2/\lambda_1 = 6{,}85$. O quociente de Rayleigh com $\mathbf{x} = (0{,}619;\ 1)$ dá $\rho = 0{,}382$, já com três algarismos corretos.

In [ ]:
K2 = np.array([[2.0, -1.0], [-1.0, 1.0]])
r = metodo_potencia(K2, x0=[1.0, 1.0], inversa=True, tol=1e-14)
for k, (x, lam) in enumerate(zip(r.historico[:4], r.historico_autovalor[:4])):
    print(f"k = {k}: x/x[1] = {x / x[1]},  quociente de Rayleigh = {lam:.6f}")
print("exatos:", np.linalg.eigvalsh(K2))

## 5. Explore

### 5.1 Os modos de vibração do prédio

Arraste o slider para ver cada modo de vibração do prédio da seção 1. A figura mostra a forma deformada (exagerada), com os andares na altura real.

In [ ]:
w2, modos = sl.eigh(K, M)                       # problema generalizado K v = ω² M v
freq = np.sqrt(w2) / (2 * np.pi)
alturas = np.arange(0, 5) * 3.0                 # 3 m por andar

def figura_modo(j):
    v = modos[:, j] / np.max(np.abs(modos[:, j]))
    if v[-1] < 0:
        v = -v
    u = np.r_[0.0, v] * 2.0                       # deslocamento exagerado, chão parado
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=np.zeros(5), y=alturas, mode="lines+markers", name="prédio em repouso",
                             line=dict(color=CORES["referencia"], dash="dot"), marker=dict(size=6)))
    fig.add_trace(go.Scatter(x=u, y=alturas, mode="lines+markers", name="modo deformado",
                             line=dict(color=CORES["aproximacao"], width=4), marker=dict(size=[0, 18, 18, 18, 14], symbol="square")))
    fig.add_vline(x=0, line=dict(color=CORES["referencia"], width=1))
    fig.update_xaxes(title="deslocamento (normalizado)", range=[-3, 3])
    fig.update_yaxes(title="altura (m)", range=[-0.5, 13])
    fig.update_layout(title=f"modo {j + 1}: f = {numero(freq[j], 2)} Hz", width=520, height=480)
    return fig

quadros = [(str(j + 1), figura_modo(j)) for j in range(4)]
mostrar(com_controle(quadros, rotulo="modo ", ativo=0, titulos=[q[1].layout.title.text for q in quadros]))
print("frequências naturais (Hz):", freq.round(3))

**O que notar:** no modo 1, o fundamental, todos os andares se deslocam para o mesmo lado, cada vez mais com a altura. O modo $j$ tem $j - 1$ trocas de sinal ao longo da altura. É um padrão geral, o mesmo das cordas vibrantes: frequências mais altas, formas mais "onduladas". A frequência fundamental, $2{,}45$ Hz, está a apenas 2 % dos $2{,}5$ Hz da máquina.

### 5.2 Compressão com a SVD

Uma imagem em tons de cinza é uma matriz. Arraste o slider para reconstruí-la com os $k$ primeiros termos da SVD.

In [ ]:
# Uma imagem sintética, 120 × 180: fundo em degradê, um disco, uma placa e uma treliça
ny, nx = 120, 180
Y, X = np.mgrid[0:ny, 0:nx] / np.array([ny, nx])[:, None, None] * np.array([160, 240])[:, None, None]
img = 0.25 + 0.35 * X / 240
img[(X - 70) ** 2 + (Y - 80) ** 2 < 40**2] = 0.9
img[(Y > 30) & (Y < 70) & (X > 130) & (X < 220)] = 0.1
for c in range(130, 221, 15):
    img[(Y > 95) & (Y < 140) & (np.abs(X - c - (Y - 95) * 0.33) < 2.5)] = 0.95
img[(Y > 92) & (Y < 97) & (X > 125) & (X < 225)] = 0.95
img[(Y > 137) & (Y < 142) & (X > 125) & (X < 225)] = 0.95

U, s, Vt = np.linalg.svd(img, full_matrices=False)

def figura_posto(k):
    Ak = (U[:, :k] * s[:k]) @ Vt[:k]
    erro = np.linalg.norm(img - Ak) / np.linalg.norm(img)
    armazenado = k * (ny + nx + 1) / (ny * nx)
    fig = go.Figure(go.Heatmap(z=np.round(Ak[::-1], 2), colorscale="Gray", zmin=0, zmax=1, showscale=False))
    fig.update_xaxes(visible=False)
    fig.update_yaxes(visible=False, scaleanchor="x")
    fig.update_layout(title=f"posto {k}: {numero(100 * armazenado, 1)} % dos números, erro relativo {numero(100 * erro, 1)} %",
                      width=600, height=460)
    return fig

postos = [1, 2, 3, 5, 10, 20, 40, 80]
quadros = [(str(k), figura_posto(k)) for k in postos]
mostrar(com_controle(quadros, rotulo="posto k = ", ativo=4, titulos=[q[1].layout.title.text for q in quadros]))

In [ ]:
fig = go.Figure(go.Scatter(x=np.arange(1, len(s) + 1), y=s / s[0], mode="lines+markers", marker=dict(size=4),
                           line=dict(color=CORES["aproximacao"]), name="σ_k / σ_1"))
fig.update_xaxes(title="k")
fig.update_yaxes(type="log", title="σ_k / σ_1", exponentformat="power")
fig.update_layout(title="Os valores singulares da imagem caem depressa")
mostrar(fig)

**O que notar:** as partes da imagem alinhadas com os eixos, como o degradê e a placa retangular, são de posto baixo e aparecem com poucos termos. O disco e as diagonais da treliça precisam de muitos mais. O gráfico dos valores singulares diz, antes de qualquer reconstrução, quanto a matriz é compressível.

## 6. Trabalhando com o agente

### 6.1 O pedido e o código

> *"Tenho as matrizes de massa `M` e de rigidez `K` de um prédio. Escreva uma função que calcule a frequência fundamental (em Hz) e o modo de vibração correspondente pelo método da potência, sem usar bibliotecas de autovalores. Depois diga se há risco de ressonância com uma máquina de 2,5 Hz, com margem de 20 %."*

Um código típico:

In [ ]:
# Código devolvido pelo agente (copiado sem alterações)
def frequencia_fundamental(M, K, tol=1e-12, max_iter=10_000):
    A = np.linalg.inv(M) @ K                     # K v = ω² M v  ->  (M⁻¹K) v = ω² v
    x = np.ones(A.shape[0])
    lam = 0.0
    for _ in range(max_iter):
        y = A @ x
        x_novo = y / np.linalg.norm(y)
        lam_novo = x_novo @ A @ x_novo
        if abs(lam_novo - lam) < tol * abs(lam_novo):
            break
        x, lam = x_novo, lam_novo
    return np.sqrt(lam_novo) / (2 * np.pi), x_novo

f1_agente, modo_agente = frequencia_fundamental(M, K)
print(f"frequência fundamental: {f1_agente:.2f} Hz")
print("modo:", (modo_agente / np.abs(modo_agente).max()).round(3))
margem = abs(f1_agente - 2.5) / 2.5
print(f"distância à máquina: {100 * margem:.0f} %  ->  {'sem risco de ressonância' if margem > 0.2 else 'RISCO de ressonância'}")

O código converge, o resultado tem a unidade certa, e a conclusão é tranquilizadora. Mas ela está errada, e a margem real é de 2 %.

### 6.2 Verificação: o princípio de Rayleigh e a forma do modo

Duas verificações não precisam de nenhuma biblioteca.

**Princípio de Rayleigh.** Para qualquer forma de deslocamento $\mathbf{x}$, o quociente {eq}`eq-rayleigh` é **maior ou igual** a $\omega_1^2$. Uma forma razoável para o primeiro modo é um deslocamento que cresce linearmente com a altura:

In [ ]:
x_chute = np.array([1.0, 2.0, 3.0, 4.0])
f_rayleigh = np.sqrt((x_chute @ K @ x_chute) / (x_chute @ M @ x_chute)) / (2 * np.pi)
print(f"limite superior para a fundamental (Rayleigh): {f_rayleigh:.2f} Hz")
print(f"valor do agente: {f1_agente:.2f} Hz  ->  {'IMPOSSÍVEL: maior que o limite' if f1_agente > f_rayleigh else 'compatível'}")

A frequência fundamental **não pode** passar de $2{,}48$ Hz, e o agente encontrou $11{,}6$ Hz.

**Forma do modo.** O modo devolvido troca de sinal três vezes: o primeiro andar vai para um lado, o segundo para o outro, e assim por diante. Pela seção 5.1, isso é a forma do modo **mais alto**, não do fundamental.

### 6.3 O diagnóstico

O método da potência converge para o autovalor de **maior** módulo. O agente aplicou o método correto à pergunta errada: encontrou a frequência natural **mais alta**, $11{,}6$ Hz, e a chamou de fundamental. A fundamental é a **menor**, e exige a potência **inversa** (seção 3.3).

### 6.4 A correção

In [ ]:
def frequencia_fundamental_corrigida(M, K, tol=1e-12, max_iter=10_000):
    # Potência inversa: resolve K y = M x a cada passo (fatoração LU feita uma vez)
    fatores = sl.lu_factor(K)
    x = np.ones(K.shape[0])
    lam = 0.0
    for _ in range(max_iter):
        y = sl.lu_solve(fatores, M @ x)
        x = y / np.sqrt(y @ M @ y)              # normaliza com peso M
        lam_novo = (x @ K @ x) / (x @ M @ x)     # quociente de Rayleigh generalizado
        if abs(lam_novo - lam) < tol * lam_novo:
            break
        lam = lam_novo
    return np.sqrt(lam_novo) / (2 * np.pi), x

f1, modo1 = frequencia_fundamental_corrigida(M, K)
print(f"fundamental (potência inversa): {f1:.4f} Hz;  modo: {(modo1 / modo1[-1]).round(3)}")
print(f"fundamental (scipy.linalg.eigh): {freq[0]:.4f} Hz")
compara(f1, freq[0], rtol=1e-10)
print(f"distância à máquina: {100 * abs(f1 - 2.5) / 2.5:.1f} %  ->  RISCO de ressonância")

> **Lição.** O agente seguiu o pedido ao pé da letra, "pelo método da potência", e o método da potência responde a outra pergunta. Verifique autovalores com o **princípio de Rayleigh**, que dá limites a partir de qualquer chute, e com a **forma física** do modo. Na engenharia, a primeira pergunta é sempre: qual autovalor eu quero?

## 7. Quando falha

### 7.1 Autovalores próximos: o método da potência se arrasta

O erro do método da potência cai por $|\lambda_2/\lambda_1|$ a cada passo. Com autovalores próximos, isso é quase 1:

In [ ]:
rng = np.random.default_rng(0)
Q, _ = np.linalg.qr(rng.normal(size=(6, 6)))
resultados = {}
for razao in [0.5, 0.9, 0.99]:
    A = Q @ np.diag([1.0, razao, 0.4, 0.3, 0.2, 0.1]) @ Q.T
    r = metodo_potencia(A, x0=rng.normal(size=6), tol=1e-12, max_iter=5000)
    resultados[f"|λ₂/λ₁| = {numero(razao, 2)} ({r.iteracoes} it.)"] = r
fig = grafico_iteracoes(resultados, titulo="Método da potência: variação relativa de λ a cada iteração")
fig.update_xaxes(type="log", title="iteração (escala log)")
mostrar(fig)

Com $|\lambda_2/\lambda_1| = 0{,}99$, são mais de mil iterações. Repare também no começo dessa curva: a variação de $\lambda$ cai a $10^{-6}$ em umas dez iterações, fica parada e depois **sobe** de novo, enquanto o vetor ainda gira devagar em direção a $\mathbf{v}_1$. Um critério de parada com tolerância $10^{-6}$ teria parado ali, com o autovetor errado. É o mesmo alerta do capítulo 4: variação pequena não é erro pequeno. Deslocamentos, potência inversa e o algoritmo QR existem para contornar a lentidão.

### 7.2 O polinômio característico

Os autovalores são as raízes de $\det(A - \lambda I)$. Calcular o polinômio e depois as raízes é, porém, um problema **mal condicionado**: as raízes de um polinômio de grau alto são extremamente sensíveis aos coeficientes. Uma matriz simétrica com autovalores $1, 2, \ldots, 20$ mostra isso:

In [ ]:
n = 20
Q, _ = np.linalg.qr(rng.normal(size=(n, n)))
A = Q @ np.diag(np.arange(1.0, n + 1)) @ Q.T
pelo_polinomio = np.sort(np.roots(np.poly(A)).real)
pela_biblioteca = np.linalg.eigvalsh(A)
print("erro máximo, polinômio característico:", f"{np.abs(pelo_polinomio - np.arange(1, n + 1)).max():.1e}")
print("erro máximo, np.linalg.eigvalsh:      ", f"{np.abs(pela_biblioteca - np.arange(1, n + 1)).max():.1e}")

O problema de autovalores de uma matriz simétrica é **bem condicionado**: perturbações de tamanho $\epsilon$ na matriz mudam os autovalores em no máximo $\epsilon$. Passar pelo polinômio característico troca esse problema bom por um problema ruim. Nenhuma biblioteca faz isso.

### 7.3 Matrizes não simétricas podem ser muito sensíveis

Para matrizes não simétricas, a garantia anterior não vale. A matriz $n \times n$ com 1 na diagonal e 1 logo acima dela tem todos os autovalores iguais a 1. Uma perturbação de $10^{-10}$ num único canto muda os autovalores em $10^{-10/n}$:

In [ ]:
for n in [2, 5, 10]:
    J = np.eye(n) + np.diag(np.ones(n - 1), 1)
    J[-1, 0] = 1e-10
    print(f"n = {n:2d}: maior |λ − 1| = {np.abs(np.linalg.eigvals(J) - 1).max():.1e}   (previsão 10^(−10/n) = {10 ** (-10 / n):.1e})")

Com $n = 10$, uma perturbação na décima casa decimal muda os autovalores na **primeira**. Em problemas não simétricos, como estabilidade de escoamentos e sistemas de controle, os autovalores podem enganar, e os valores singulares e o pseudoespectro dão uma informação mais confiável.

## Resumo

| Ferramenta | Para quê | Custo / velocidade |
| --- | --- | --- |
| Método da potência | maior autovalor em módulo | $O(n^2)$ por passo; fator $\lvert\lambda_2/\lambda_1\rvert$ |
| Potência inversa (com deslocamento $\sigma$) | menor autovalor, ou o mais próximo de $\sigma$ | uma LU + $O(n^2)$ por passo |
| Quociente de Rayleigh | estimar $\lambda$ a partir de um vetor | erro $\epsilon^2$; limite superior para $\lambda_{\min}$ |
| Algoritmo QR (`eigh`, `eig`) | todos os autovalores | $O(n^3)$ |
| Lanczos / Arnoldi (`eigsh`) | alguns autovalores de matrizes esparsas | por produto matriz-vetor |
| SVD | posto, condicionamento, aproximação de posto baixo, PCA | $O(mn^2)$ |

## Exercícios

**Conceituais (sem computador)**

1. Mostre que os autovalores de uma matriz simétrica real são reais e que autovetores de autovalores diferentes são ortogonais.
2. Prove o princípio de Rayleigh para uma matriz simétrica, escrevendo $\mathbf{x}$ na base de autovetores.
3. Para o prédio de dois andares da seção 4, mostre que os modos são $M$-ortogonais.
4. Mostre que os valores singulares de $A$ são as raízes dos autovalores de $A^{\mathsf T}A$, e que $\kappa_2(A) = \sigma_1/\sigma_n$.

**Verificação (com agente)**

5. Peça a um agente o algoritmo QR **sem** deslocamentos. Verifique com uma matriz de autovalores conhecidos, meça a velocidade de convergência dos elementos abaixo da diagonal e relacione com as razões $\lambda_{i+1}/\lambda_i$.
6. Peça a um agente uma análise de componentes principais (PCA) de uma tabela de dados de motores (potência, cilindrada, massa, consumo). Verifique se o código centraliza e padroniza as colunas: o que acontece com a primeira componente se a massa estiver em gramas em vez de quilogramas?
7. **PageRank.** O "valor" de cada página da web é o autovetor dominante de uma matriz de transição. Peça a um agente o PageRank de um grafo de 8 páginas pelo método da potência e verifique com `np.linalg.eig`. Por que o método da potência converge rapidamente para essa matriz?

**Projeto aberto**

8. **Amortecedor de massa sintonizada.** Acrescente ao telhado do prédio uma pequena massa ligada por uma mola, sintonizada na frequência fundamental. Calcule as novas frequências naturais e a resposta do prédio a uma força harmônica de 2,5 Hz no telhado, com e sem o amortecedor.

## Leituras

- S. C. Chapra e R. P. Canale, *Métodos Numéricos para Engenharia*, capítulo 27 (seção de autovalores).
- L. N. Trefethen e D. Bau, *Numerical Linear Algebra*, SIAM, 1997, aulas 4 e 5 (SVD) e 24 a 29 (autovalores).
- A. K. Chopra, *Dynamics of Structures*, Pearson, capítulos 10 e 11 (modos de vibração de prédios).